# Retrieval Experiments

use this notebook to compare chunk sizes, overlap, top-k, hybrid alpha, and embedding models against a fixed local benchmark set.

In [ ]:
import itertools, json, httpx, pandas as pd

BASE_URL = 'http://127.0.0.1:8000'
questions = [
    'What are the main claims in the indexed documents?',
    'Which source best supports the conclusion?'
]
grid = list(itertools.product([4, 6, 10], [0.35, 0.62, 0.85]))
rows = []
for top_k, hybrid_alpha in grid:
    for question in questions:
        payload = {'query': question, 'top_k': top_k, 'hybrid_alpha': hybrid_alpha}
        response = httpx.post(f'{BASE_URL}/api/search', json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        rows.append({
            'question': question,
            'top_k': top_k,
            'hybrid_alpha': hybrid_alpha,
            'latency_ms': data['latency_ms'],
            'hits': len(data['hits']),
            'top_score': data['hits'][0]['score'] if data['hits'] else 0,
        })
pd.DataFrame(rows)